# Week 3: Clean the Data and Answer Questions
**Group 10: Space Observation and Satellite Data Trends**

This notebook applies the fixes planned in `02_exploration.ipynb`, saves a processed dataset and answers our five research questions.

In [1]:
import pandas as pd
import numpy as np   # numpy gives us NaN (missing value) and np.where (if/else on a whole column)

# Show every column instead of hiding the middle ones with "..."
pd.set_option("display.max_columns", None)

# Where to read the raw data from and where to save the cleaned data
RAW_PATH = "../data/raw/star_classification.csv"
PROCESSED_PATH = "../data/processed/star_classification_clean.csv"

# Read the IDs as text so they are not rounded (see Week 2)
raw = pd.read_csv(RAW_PATH, dtype={"obj_ID": str, "spec_obj_ID": str})

# Work on a copy so the original "raw" table is never changed,
# which lets us compare before and after at the end
df = raw.copy()
print("Raw shape:", raw.shape)

Raw shape: (100000, 18)


## Part A: Cleaning

### Step 1: Hidden missing values (-9999)
**Decision: convert -9999 to missing (NaN), then drop the affected row.**
Reason: -9999 is SDSS's code for "no measurement". Only one row is affected, and it is missing three of its five brightness bands, so filling them would mean inventing most of that object's brightness. Dropping 1 row out of 100,000 has no meaningful effect on the results.

In [2]:
# The five brightness columns
bands = ["u", "g", "r", "i", "z"]

# Count the -9999 values before changing anything (used in the summary at the end)
sentinel_before = int((df[bands] == -9999).sum().sum())
print("-9999 values before:", sentinel_before)

# Replace -9999 with NaN so pandas treats it as missing instead of a real number
df[bands] = df[bands].replace(-9999, np.nan)
print("Missing values per band after conversion:")
print(df[bands].isnull().sum())

# Show which row is about to be removed, so the decision is visible
dropped_row = df[df[bands].isnull().any(axis=1)]
print()
print("Row being dropped:")
print(dropped_row[["spec_obj_ID", "class", "u", "g", "r", "i", "z", "redshift"]])

# Remove rows that have a missing value in any of the five bands
df = df.dropna(subset=bands)
print()
print("Rows after dropping:", len(df))

-9999 values before: 3
Missing values per band after conversion:
u    1
g    1
r    0
i    0
z    1
dtype: int64

Row being dropped:
               spec_obj_ID class   u   g        r         i   z  redshift
79543  3731276904880250880  STAR NaN NaN  18.1656  18.01675 NaN  0.000089

Rows after dropping: 99999


### Step 2: Remove a column with no information
**Decision: drop `rerun_ID`.** It is 301 in every row (shown in Week 2), so it cannot help answer any question.

In [3]:
# Confirm the column only has one value before deleting it
print("Unique rerun_ID values:", df["rerun_ID"].unique())
df = df.drop(columns=["rerun_ID"])
print("Columns left:", df.shape[1])

Unique rerun_ID values: [301]
Columns left: 17


### Step 3: Identifiers and duplicates
**Decision: use `spec_obj_ID` as the unique key; keep `obj_ID` but flag it as unreliable; keep all rows.**
Reason: there are no fully duplicated rows, and every `spec_obj_ID` is unique, so every row is a separate measurement. `obj_ID` values repeat because the source file rounded them, not because the same object was recorded twice, so we must not remove rows based on `obj_ID`.

In [4]:
# 0 here means no row is an exact copy of another, so nothing needs removing
print("Fully duplicated rows:", df.duplicated().sum())
# 0 here confirms spec_obj_ID can be used as a unique ID for every row
print("Repeated spec_obj_ID:", df["spec_obj_ID"].duplicated().sum())

# New True/False column: True if this row's obj_ID also appears in another row
df["obj_ID_shared"] = df["obj_ID"].duplicated(keep=False)
print("Rows whose obj_ID is shared with another row (flagged):", df["obj_ID_shared"].sum())

Fully duplicated rows: 0


Repeated spec_obj_ID: 0
Rows whose obj_ID is shared with another row (flagged): 33183


### Step 4: Normalise category text
**Decision: strip spaces and convert `class` to upper case.**
Reason: Week 2 found only three clean labels, but this guards against variations like " star" or "Galaxy" if the data is ever refreshed. We count how many values actually change.

In [5]:
# Keep a copy of the labels so we can count how many change
class_before = df["class"].copy()

# strip() removes spaces at the start/end; upper() makes everything capitals
df["class"] = df["class"].str.strip().str.upper()

# Compare old and new labels and count the differences
class_changed = int((class_before != df["class"]).sum())
print("Class values changed:", class_changed)
print("Labels now:", sorted(df["class"].unique()))

Class values changed: 0
Labels now: ['GALAXY', 'QSO', 'STAR']


### Step 5: Fix types and add date columns
**Decision: convert `MJD` into a real date, and add `obs_year` and `obs_month`.** Reason: we need dates to compare observations over time (Question 3). `MJD` is kept so the original value is still available.

In [6]:
# Convert the MJD day count into a calendar date (days counted from 17 Nov 1858)
df["obs_date"] = pd.to_datetime(df["MJD"], unit="D", origin=pd.Timestamp("1858-11-17"))

# Pull the year and month out of the date into their own columns
df["obs_year"] = df["obs_date"].dt.year
df["obs_month"] = df["obs_date"].dt.month

print("Dates that failed to parse:", df["obs_date"].isnull().sum())
print("Range:", df["obs_date"].min().date(), "to", df["obs_date"].max().date())
df[["MJD", "obs_date", "obs_year", "obs_month"]].head()

Dates that failed to parse: 0
Range: 2000-03-05 to 2020-03-24


,MJD,obs_date,obs_year,obs_month
0,56354,2013-03-03,2013,3
1,58158,2018-02-09,2018,2
2,55592,2011-01-31,2011,1
3,58039,2017-10-13,2017,10
4,56187,2012-09-17,2012,9


### Step 6: Add sky-region columns
**Decision: group right ascension (`alpha`) into six 60-degree bands, and split declination (`delta`) into north and south of the celestial equator.** Reason: positions are continuous numbers, so grouping them lets us compare regions of the sky in Question 4.

In [7]:
# The six band names, in sky order, reused below to display results in this order
ra_order = ["0-60", "60-120", "120-180", "180-240", "240-300", "300-360"]

# pd.cut puts each alpha value into the 60-degree band it falls in.
# include_lowest=True makes sure a value of exactly 0 is included in the first band.
df["ra_band"] = pd.cut(
    df["alpha"],
    bins=[0, 60, 120, 180, 240, 300, 360],
    labels=ra_order,
    include_lowest=True,
).astype(str)   # stored as text so it saves cleanly to CSV

# np.where works like if/else on every row: North if delta >= 0, otherwise South
df["hemisphere"] = np.where(df["delta"] >= 0, "North", "South")

# [ra_order] puts the counts in sky order instead of the order pandas picks
print(df["ra_band"].value_counts()[ra_order])
print()
print(df["hemisphere"].value_counts())

ra_band
0-60       17589
60-120      3918
120-180    28110
180-240    28394
240-300     7974
300-360    14014
Name: count, dtype: int64

hemisphere
North    87939
South    12060
Name: count, dtype: int64


### Step 7: Flag outliers (do not delete)
**Decision: add flag columns instead of removing rows.**
- `negative_redshift`: redshift below 0. For stars this is physically real (moving towards us), so these rows are kept.
- `suspicious_galaxy_z`: galaxies with negative redshift, which are unexpected and worth treating with caution.
- `qso_high_z_check`: quasars with redshift above 6. Quasars this far away are extremely rare, yet the values here bunch up just below 7.01 (six share exactly 7.011245), which looks like the upper edge of the SDSS fitting software rather than real distances. We keep them but treat them with caution.
- `redshift_outlier` and `r_outlier`: values outside 1.5 x IQR **within each class**. We use each class separately because quasars are naturally much further away than stars, so one cut-off for everyone would wrongly flag almost all quasars.

In [8]:
def iqr_outlier(values):
    """True for values more than 1.5 x IQR outside the middle 50%."""
    q1 = values.quantile(0.25)   # 25% of values are below this
    q3 = values.quantile(0.75)   # 75% of values are below this
    iqr = q3 - q1                # the spread of the middle 50%
    # Standard rule: anything more than 1.5 x IQR beyond q1 or q3 is an outlier
    return (values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)


# Each flag is a True/False column; nothing is deleted
df["negative_redshift"] = df["redshift"] < 0
df["suspicious_galaxy_z"] = (df["class"] == "GALAXY") & (df["redshift"] < 0)
df["qso_high_z_check"] = (df["class"] == "QSO") & (df["redshift"] > 6)

# groupby("class").transform(...) runs iqr_outlier separately for stars, galaxies
# and quasars, so each class is compared only with itself
df["redshift_outlier"] = df.groupby("class")["redshift"].transform(iqr_outlier)
df["r_outlier"] = df.groupby("class")["r"].transform(iqr_outlier)

# Count how many rows of each class got each flag (True counts as 1)
flags = ["negative_redshift", "suspicious_galaxy_z", "qso_high_z_check",
         "redshift_outlier", "r_outlier", "obj_ID_shared"]
df.groupby("class")[flags].sum()

,negative_redshift,suspicious_galaxy_z,qso_high_z_check,redshift_outlier,r_outlier,obj_ID_shared
class,,,,,,
GALAXY,42,42,0,254,35,18633
QSO,0,0,118,362,352,7058
STAR,13682,0,0,1599,29,7492


### Step 8: Save the processed dataset and prove it still meets the minimum

In [9]:
# Save the cleaned table; index=False stops pandas adding an extra row-number column
df.to_csv(PROCESSED_PATH, index=False)

# Read the saved file back in to prove what was actually written to disk
check = pd.read_csv(PROCESSED_PATH)
rows, cols = check.shape
print("Processed shape:", check.shape)
print("At least 15,000 rows:", rows >= 15000)
print("At least 10 columns:", cols >= 10)
print(check.columns.tolist())

Processed shape: (99999, 28)
At least 15,000 rows: True
At least 10 columns: True
['obj_ID', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'run_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'class', 'redshift', 'plate', 'MJD', 'fiber_ID', 'obj_ID_shared', 'obs_date', 'obs_year', 'obs_month', 'ra_band', 'hemisphere', 'negative_redshift', 'suspicious_galaxy_z', 'qso_high_z_check', 'redshift_outlier', 'r_outlier']


### Step 9: Before/after cleaning summary

In [10]:
# Side-by-side table comparing the raw data with the cleaned data.
# "-" means the measure only applies after cleaning.
summary = pd.DataFrame({
    "measure": [
        "Rows",
        "Columns",
        "Null values (incl. -9999 placeholders)",
        "Fully duplicated rows",
        "Class labels changed by normalising",
        "Columns dropped",
        "Columns added (dates, regions, flags)",
        "Rows flagged as outliers (redshift or r)",
        "Quasars flagged for implausibly high redshift",
    ],
    "before": [
        raw.shape[0],
        raw.shape[1],
        int(raw.isnull().sum().sum()) + sentinel_before,
        int(raw.duplicated().sum()),
        "-",
        "-",
        "-",
        "-",
        "-",
    ],
    "after": [
        df.shape[0],
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        class_changed,
        1,
        # columns now, minus the original columns that are left after dropping rerun_ID
        df.shape[1] - (raw.shape[1] - 1),
        # | means OR: rows flagged by either outlier check
        int((df["redshift_outlier"] | df["r_outlier"]).sum()),
        int(df["qso_high_z_check"].sum()),
    ],
})
print(f"Rows retained: {len(df):,} of {len(raw):,} ({len(df) / len(raw) * 100:.3f}%)")
summary

Rows retained: 99,999 of 100,000 (99.999%)


,measure,before,after
0,Rows,100000,99999
1,Columns,18,28
2,Null values (incl. -9999 placeholders),3,0
3,Fully duplicated rows,0,0
4,Class labels changed by normalising,-,0
5,Columns dropped,-,1
6,"Columns added (dates, regions, flags)",-,11
7,Rows flagged as outliers (redshift or r),-,2614
8,Quasars flagged for implausibly high redshift,-,118


## Part B: Answering the research questions

### Q1. What share of the sample is each class?

In [11]:
# Count per class, and the same as a percentage of all rows
q1 = pd.DataFrame({
    "count": df["class"].value_counts(),
    "share_pct": (df["class"].value_counts(normalize=True) * 100).round(2),
})
q1

,count,share_pct
class,,
GALAXY,59445,59.45
STAR,21593,21.59
QSO,18961,18.96


**Interpretation:** Galaxies make up almost 60% of the cleaned sample (59,445 objects), about 2.75 times as many as stars (21,593) and 3 times as many as quasars (18,961). This reflects what SDSS chose to observe rather than how common each type is in the universe. Because the classes are so unequal, later comparisons use averages and percentages instead of raw totals.

### Q2. How does redshift (distance) differ by class?

In [12]:
# For each class: number of objects and the mean, median, lowest and highest redshift
q2 = df.groupby("class")["redshift"].agg(["count", "mean", "median", "min", "max"]).round(4)
# Sort so the most distant class (highest mean) comes first
q2.sort_values("mean", ascending=False)

,count,mean,median,min,max
class,,,,,
QSO,18961,1.7197,1.6172,0.0005,7.0112
GALAXY,59445,0.4216,0.4563,-0.0100,1.9955
STAR,21593,-0.0001,-0.0001,-0.0041,0.0042


In [13]:
# Filtering: keep only the extremely distant objects (redshift above 4)
very_far = df[df["redshift"] > 4]
print("Objects with redshift > 4:", len(very_far))
print(very_far["class"].value_counts())
# How many of those are the questionable quasars flagged in Step 7
print("...of which flagged as implausibly high (> 6):", very_far["qso_high_z_check"].sum())
print()

# Sorting: highest redshift first, then show the top 5
print("Five most distant objects:")
df.sort_values("redshift", ascending=False)[["spec_obj_ID", "class", "redshift", "r"]].head(5)

Objects with redshift > 4: 321
class
QSO    321
Name: count, dtype: int64
...of which flagged as implausibly high (> 6): 118

Five most distant objects:


,spec_obj_ID,class,redshift,r
39650,10294352840878217216,QSO,7.011245,22.19393
51922,12741906418535520256,QSO,7.011245,21.78374
6178,9452082406616225792,QSO,7.011245,21.51426
71744,7419807210052802560,QSO,7.011245,20.29786
53375,7237420221337982976,QSO,7.011245,19.98343


**Interpretation:** Quasars are by far the most distant class, with a median redshift of 1.62 compared with 0.46 for galaxies. Stars have a median of essentially zero because they are inside our own galaxy. Every object with redshift above 4 is a quasar, which shows that redshift alone almost separates the three classes. We use the median rather than the mean because the maximum (7.01) comes from the flagged high-redshift quasars, whose values are probably fitting errors.

### Q3. How many records were observed per year, and has the class mix changed?

In [14]:
# crosstab counts rows for every year/class combination.
# margins=True adds a Total row and column.
q3_counts = pd.crosstab(df["obs_year"], df["class"], margins=True, margins_name="Total")
q3_counts

class,GALAXY,QSO,STAR,Total
obs_year,,,,
2000,988,114,186,1288
2001,2316,359,462,3137
2002,2571,284,866,3721
2003,3820,346,1297,5463
2004,1939,212,1041,3192
2005,2058,182,908,3148
2006,1971,192,1085,3248
2007,1469,159,1452,3080
2008,1321,166,2422,3909


In [15]:
# normalize="index" turns each year's counts into fractions of that year's total;
# x100 makes them percentages, so each row adds up to 100
q3_share = (pd.crosstab(df["obs_year"], df["class"], normalize="index") * 100).round(1)
q3_share

class,GALAXY,QSO,STAR
obs_year,,,
2000,76.7,8.9,14.4
2001,73.8,11.4,14.7
2002,69.1,7.6,23.3
2003,69.9,6.3,23.7
2004,60.7,6.6,32.6
2005,65.4,5.8,28.8
2006,60.7,5.9,33.4
2007,47.7,5.2,47.1
2008,33.8,4.2,62.0


In [16]:
# Date-based comparison: label each row with its decade, then compare the class mix
df["period"] = np.where(df["obs_year"] <= 2009, "2000-2009", "2010-2020")
period_mix = (pd.crosstab(df["period"], df["class"], normalize="index") * 100).round(1)
# Add the number of records in each period for context
period_mix["records"] = df["period"].value_counts()
period_mix

class,GALAXY,QSO,STAR,records
period,,,,
2000-2009,57.6,6.5,35.9,32253
2010-2020,60.3,24.9,14.8,67746


**Interpretation:** Observations peak in 2010 to 2013 (around 8,000 to 11,000 per year) and drop sharply in 2019 and 2020, when the data release ends. The class mix changes a lot: from 2000 to 2009, quasars are only about 3 to 11% of each year's records, but from 2014 onwards they are 31 to 46%. Stars also spike to 90% of records in 2009. These shifts most likely reflect changes in what each phase of the survey targeted, not changes in the sky, so year-to-year counts should not be read as natural trends.

### Q4. Where in the sky are the objects, and do the classes cluster differently?

In [17]:
# Count of each class in each 60-degree sky band, plus totals
q4_counts = pd.crosstab(df["ra_band"], df["class"], margins=True, margins_name="Total")
# reindex puts the rows in sky order (0-60 first) with Total at the bottom
q4_counts.reindex(ra_order + ["Total"])

class,GALAXY,QSO,STAR,Total
ra_band,,,,
0-60,10100,3342,4147,17589
60-120,1369,668,1881,3918
120-180,17926,5082,5102,28110
180-240,17998,5441,4955,28394
240-300,4132,1521,2321,7974
300-360,7920,2907,3187,14014
Total,59445,18961,21593,99999


In [18]:
# Percentage of each class within each band (each row adds up to 100)
q4_share = (pd.crosstab(df["ra_band"], df["class"], normalize="index") * 100).round(1)
q4_share.reindex(ra_order)

class,GALAXY,QSO,STAR
ra_band,,,
0-60,57.4,19.0,23.6
60-120,34.9,17.0,48.0
120-180,63.8,18.1,18.2
180-240,63.4,19.2,17.5
240-300,51.8,19.1,29.1
300-360,56.5,20.7,22.7


In [19]:
# Count of each class in the northern and southern sky
q4_hemi = pd.crosstab(df["hemisphere"], df["class"], margins=True, margins_name="Total")
q4_hemi

class,GALAXY,QSO,STAR,Total
hemisphere,,,,
North,52115,17246,18578,87939
South,7330,1715,3015,12060
Total,59445,18961,21593,99999


**Interpretation:** Over half of all objects (about 56,500) lie between 120 and 240 degrees of right ascension, and 88% are in the northern sky, which matches SDSS being a northern-hemisphere telescope. The 60 to 120 degree band is the least observed (3,918 objects) and is the only band where stars outnumber galaxies (48% stars). This band likely looks closer to the plane of our own Milky Way, where stars are densest. Quasar share stays steady at 17 to 21% in every band.

### Q5. Which class is brightest or faintest on average in the r band?

Astronomical magnitudes work backwards: a **lower** number means a **brighter** object.

In [20]:
# Mean, median, min and max r-band magnitude for each class
q5 = df.groupby("class")["r"].agg(["mean", "median", "min", "max"]).round(2)
# Sort by mean so the brightest class (lowest number) comes first
q5.sort_values("mean")

,mean,median,min,max
class,,,,
STAR,18.95,18.96,9.82,29.37
GALAXY,19.59,20.10,11.75,29.57
QSO,20.62,20.77,12.36,27.40


In [21]:
# Average magnitude per class in all five bands, to check r is not unusual
print("Average magnitude in all five bands:")
df.groupby("class")[bands].mean().round(2)

Average magnitude in all five bands:


,u,g,r,i,z
class,,,,,
GALAXY,22.59,20.91,19.59,18.85,18.45
QSO,21.55,20.93,20.62,20.43,20.27
STAR,21.15,19.62,18.95,18.54,18.33


In [22]:
# Sort by r from lowest (brightest) to highest and show the top 10
print("Ten brightest objects in the r band:")
df.sort_values("r")[["spec_obj_ID", "class", "r", "redshift"]].head(10)

Ten brightest objects in the r band:


,spec_obj_ID,class,r,redshift
14498,3198781215817623552,STAR,9.82207,0.000051
4472,3383434854984163328,STAR,10.06854,0.000025
75784,3383404618414399488,STAR,10.11604,0.000008
75620,3383416163286491136,STAR,10.19460,0.000003
85177,3377874349250945024,STAR,10.80343,-0.000009
85285,3377753677849796608,STAR,10.86379,-0.000159
75583,3383403518902771712,STAR,10.98255,0.000174
76986,3383362836972544000,STAR,11.09069,0.000049
76321,3383414788896956416,STAR,11.35229,0.000070
15666,3198790011910645760,STAR,11.64166,0.000158


**Interpretation:** Stars are the brightest class on average (mean r = 18.95), galaxies are next (19.59) and quasars are the faintest (20.62), even though quasars are the most powerful objects. This is consistent with Q2: quasars are so far away that their light is much dimmer by the time it reaches us. Stars are bright simply because they are in our own galaxy, and they are the brightest class in all five bands.